Exploration & Data loading

1.1 Environment Setup and Library Imports

In [1]:
# --- 1. Install Required Libraries ---
# Using --quiet to keep the output clean
#!pip install torch torch_geometric pandas numpy scikit-learn networkx plotly --quiet
# pyg-lib is a dependency for optimized operations in PyTorch Geometric, especially for NeighborLoader.
# The installation command is tailored to the PyTorch version for compatibility.
#!pip install pyg-lib -f https://data.pyg.org/whl/torch-$(python -c 'import torch; print(torch.__version__)').html --quiet

# --- 2. Import All Necessary Modules ---

# Core Libraries
import os
import random
import numpy as np
import pandas as pd

# PyTorch & PyTorch Geometric (PyG)
import torch
import torch.nn as nn
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import HeteroConv, SAGEConv
from torch.nn import Linear

# Machine Learning & Evaluation
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, roc_curve

# Visualization
import networkx as nx
import plotly.graph_objects as go
from IPython.display import display

# --- 3. Set Seeds for Reproducibility ---

def set_seeds(seed=42):
    """
    Sets random seeds for all relevant libraries to ensure reproducibility of results.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # The following two lines are for ensuring deterministic behavior on CUDA.
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"All random seeds set to {seed}.")

# Set the seeds for the entire notebook session
set_seeds(42)

# --- 4. Configure Device (GPU or CPU) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 5. Configure Pandas Display Options ---
# This helps in inspecting DataFrames more effectively.
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
print("Pandas display options configured.")

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All random seeds set to 42.
Using device: cpu
Pandas display options configured.


1.2 Define Hyperparameters

In [2]:
# --- Hyperparameters ---
# These parameters are defined globally to be easily accessible and modifiable.
# The values are chosen based on the defaults for the 'BlogCatalog' dataset
# in the original SL-GAD paper's implementation.

# --- Data and Loader Settings ---
SUBGRAPH_SIZE = 4
BATCH_SIZE = 300

# --- Model Settings ---
EMBEDDING_DIM = 64

# --- Training Settings ---
LEARNING_RATE = 0.003  # Default for BlogCatalog from the paper
WEIGHT_DECAY = 0.0
# Set a lower number of epochs for faster initial runs in the notebook.
# The paper uses 400 for BlogCatalog.
NUM_EPOCHS = 100

# --- SL-GAD Specific Loss Weights ---
ALPHA = 1.0  # Weight for the contrastive loss component
BETA = 0.6   # Weight for the generative (reconstruction) loss component

# --- Inference Settings ---
# Set a lower number of rounds for faster inference during development.
# The paper uses 256.
AUC_TEST_ROUNDS = 64

print("Hyperparameters defined:")
print(f"  - SUBGRAPH_SIZE: {SUBGRAPH_SIZE}")
print(f"  - BATCH_SIZE: {BATCH_SIZE}")
print(f"  - EMBEDDING_DIM: {EMBEDDING_DIM}")
print(f"  - LEARNING_RATE: {LEARNING_RATE}")
print(f"  - NUM_EPOCHS: {NUM_EPOCHS}")
print(f"  - ALPHA (Contrastive Weight): {ALPHA}")
print(f"  - BETA (Generative Weight): {BETA}")
print(f"  - AUC_TEST_ROUNDS: {AUC_TEST_ROUNDS}")

Hyperparameters defined:
  - SUBGRAPH_SIZE: 4
  - BATCH_SIZE: 300
  - EMBEDDING_DIM: 64
  - LEARNING_RATE: 0.003
  - NUM_EPOCHS: 100
  - ALPHA (Contrastive Weight): 1.0
  - BETA (Generative Weight): 0.6
  - AUC_TEST_ROUNDS: 64


1.3 Data Loading and Initial Inspection

In [9]:
# Define the path to the dataset.
# This path assumes the dataset is located in the default Kaggle input directory.
file_path = 'E:\\SCHOOL\\Health Insurance Fraud Detection\\data\\raw\\DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv'

# Load the dataset into a pandas DataFrame using a try-except block for robustness.
try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully.")
    print(f"Shape of the initial dataset: {df.shape}")
except FileNotFoundError:
    print(f"Error: The file was not found at {file_path}")
    print("Please ensure the dataset is correctly added to your Kaggle notebook environment.")
    df = None # Ensure df is None if loading fails

# --- Perform Initial Inspection if the DataFrame was loaded ---
if df is not None:
    print("\n--- DataFrame Info ---")
    df.info()

    print("\n\n--- First 5 Rows ---")
    display(df.head())

    print("\n\n--- Descriptive Statistics (including categorical) ---")
    # Using include='all' provides a more comprehensive overview.
    display(df.describe(include='all'))

    print("\n\n--- Unique Value Counts per Column ---")
    display(df.nunique())

Dataset loaded successfully.
Shape of the initial dataset: (66773, 81)

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66773 entries, 0 to 66772
Data columns (total 81 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   DESYNPUF_ID                     66773 non-null  object 
 1   CLM_ID                          66773 non-null  int64  
 2   SEGMENT                         66773 non-null  int64  
 3   CLM_FROM_DT                     66705 non-null  float64
 4   CLM_THRU_DT                     66705 non-null  float64
 5   PRVDR_NUM                       66773 non-null  object 
 6   CLM_PMT_AMT                     66773 non-null  float64
 7   NCH_PRMRY_PYR_CLM_PD_AMT        66773 non-null  float64
 8   AT_PHYSN_NPI                    66100 non-null  float64
 9   OP_PHYSN_NPI                    39058 non-null  float64
 10  OT_PHYSN_NPI                    7683 non-null   float64
 11

,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,OT_PHYSN_NPI,CLM_ADMSN_DT,ADMTNG_ICD9_DGNS_CD,CLM_PASS_THRU_PER_DIEM_AMT,NCH_BENE_IP_DDCTBL_AMT,NCH_BENE_PTA_COINSRNC_LBLTY_AM,NCH_BENE_BLOOD_DDCTBL_LBLTY_AM,CLM_UTLZTN_DAY_CNT,NCH_BENE_DSCHRG_DT,CLM_DRG_CD,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_DGNS_CD_3,ICD9_DGNS_CD_4,ICD9_DGNS_CD_5,ICD9_DGNS_CD_6,ICD9_DGNS_CD_7,ICD9_DGNS_CD_8,ICD9_DGNS_CD_9,ICD9_DGNS_CD_10,ICD9_PRCDR_CD_1,ICD9_PRCDR_CD_2,ICD9_PRCDR_CD_3,ICD9_PRCDR_CD_4,ICD9_PRCDR_CD_5,ICD9_PRCDR_CD_6,HCPCS_CD_1,HCPCS_CD_2,HCPCS_CD_3,HCPCS_CD_4,HCPCS_CD_5,HCPCS_CD_6,HCPCS_CD_7,HCPCS_CD_8,HCPCS_CD_9,HCPCS_CD_10,HCPCS_CD_11,HCPCS_CD_12,HCPCS_CD_13,HCPCS_CD_14,HCPCS_CD_15,HCPCS_CD_16,HCPCS_CD_17,HCPCS_CD_18,HCPCS_CD_19,HCPCS_CD_20,HCPCS_CD_21,HCPCS_CD_22,HCPCS_CD_23,HCPCS_CD_24,HCPCS_CD_25,HCPCS_CD_26,HCPCS_CD_27,HCPCS_CD_28,HCPCS_CD_29,HCPCS_CD_30,HCPCS_CD_31,HCPCS_CD_32,HCPCS_CD_33,HCPCS_CD_34,HCPCS_CD_35,HCPCS_CD_36,HCPCS_CD_37,HCPCS_CD_38,HCPCS_CD_39,HCPCS_CD_40,HCPCS_CD_41,HCPCS_CD_42,HCPCS_CD_43,HCPCS_CD_44,HCPCS_CD_45
0,00013D2EFD8E45D1,196661176988405,1,20100312.0,20100313.0,2600GD,4000.0,0.0,3.139084e+09,NaN,NaN,20100312,4580,0.0,1100.0,0.0,0.0,1.0,20100313,217,7802,78820,V4501,4280,2720,4019,V4502,73300,E9330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00016F745862898F,196201177000368,1,20090412.0,20090418.0,3900MB,26000.0,0.0,6.476809e+09,NaN,NaN,20090412,7866,0.0,1068.0,0.0,0.0,6.0,20090418,201,1970,4019,5853,7843,2768,71590,2724,19889,5849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00016F745862898F,196661177015632,1,20090831.0,20090902.0,3900HM,5000.0,0.0,6.119985e+08,6.119985e+08,NaN,20090831,6186,0.0,1068.0,0.0,0.0,2.0,20090902,750,6186,2948,56400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7092.0,6186,V5866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00016F745862898F,196091176981058,1,20090917.0,20090920.0,3913XU,5000.0,0.0,4.971603e+09,NaN,1.119000e+09,20090917,29590,0.0,1068.0,0.0,0.0,3.0,20090920,883,29623,30390,71690,34590,V1581,32723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00016F745862898F,196261176983265,1,20100626.0,20100701.0,3900MB,16000.0,0.0,6.408400e+09,1.960860e+09,NaN,20100626,5849,0.0,1100.0,0.0,0.0,5.0,20100701,983,3569,4019,3542,V8801,78820,2639,7840,7856,4271,NaN,NaN,E8889,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN




--- Descriptive Statistics (including categorical) ---


,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,OT_PHYSN_NPI,CLM_ADMSN_DT,ADMTNG_ICD9_DGNS_CD,CLM_PASS_THRU_PER_DIEM_AMT,NCH_BENE_IP_DDCTBL_AMT,NCH_BENE_PTA_COINSRNC_LBLTY_AM,NCH_BENE_BLOOD_DDCTBL_LBLTY_AM,CLM_UTLZTN_DAY_CNT,NCH_BENE_DSCHRG_DT,CLM_DRG_CD,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_DGNS_CD_3,ICD9_DGNS_CD_4,ICD9_DGNS_CD_5,ICD9_DGNS_CD_6,ICD9_DGNS_CD_7,ICD9_DGNS_CD_8,ICD9_DGNS_CD_9,ICD9_DGNS_CD_10,ICD9_PRCDR_CD_1,ICD9_PRCDR_CD_2,ICD9_PRCDR_CD_3,ICD9_PRCDR_CD_4,ICD9_PRCDR_CD_5,ICD9_PRCDR_CD_6,HCPCS_CD_1,HCPCS_CD_2,HCPCS_CD_3,HCPCS_CD_4,HCPCS_CD_5,HCPCS_CD_6,HCPCS_CD_7,HCPCS_CD_8,HCPCS_CD_9,HCPCS_CD_10,HCPCS_CD_11,HCPCS_CD_12,HCPCS_CD_13,HCPCS_CD_14,HCPCS_CD_15,HCPCS_CD_16,HCPCS_CD_17,HCPCS_CD_18,HCPCS_CD_19,HCPCS_CD_20,HCPCS_CD_21,HCPCS_CD_22,HCPCS_CD_23,HCPCS_CD_24,HCPCS_CD_25,HCPCS_CD_26,HCPCS_CD_27,HCPCS_CD_28,HCPCS_CD_29,HCPCS_CD_30,HCPCS_CD_31,HCPCS_CD_32,HCPCS_CD_33,HCPCS_CD_34,HCPCS_CD_35,HCPCS_CD_36,HCPCS_CD_37,HCPCS_CD_38,HCPCS_CD_39,HCPCS_CD_40,HCPCS_CD_41,HCPCS_CD_42,HCPCS_CD_43,HCPCS_CD_44,HCPCS_CD_45
count,66773,6.677300e+04,66773.000000,6.670500e+04,6.670500e+04,66773,66773.000000,66773.000000,6.610000e+04,3.905800e+04,7.683000e+03,6.677300e+04,66174,66773.000000,64595.000000,66773.000000,66773.000000,66705.000000,6.677300e+04,66773,66678,66247,65492,64005,61639,58369,54407,49947,45032,5455,38231.000000,22733,14462,9377,6446,4622,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
unique,37780,NaN,NaN,NaN,NaN,2675,NaN,NaN,NaN,NaN,NaN,NaN,2285,NaN,NaN,NaN,NaN,NaN,NaN,739,2740,2961,2909,2907,2915,2843,2823,2696,2571,1134,NaN,1992,1680,1370,1125,927,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,CD57573A5B77AAFE,NaN,NaN,NaN,NaN,23006G,NaN,NaN,NaN,NaN,NaN,NaN,78605,NaN,NaN,NaN,NaN,NaN,NaN,882,486,4019,4019,4019,4019,4019,4019,4019,4019,4019,NaN,4019,4019,4019,4019,4019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,14,NaN,NaN,NaN,NaN,772,NaN,NaN,NaN,NaN,NaN,NaN,2739,NaN,NaN,NaN,NaN,NaN,NaN,282,2453,4170,3830,3469,3126,2687,2329,1943,1649,170,NaN,1463,810,467,308,215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,1.965017e+14,1.001018,2.008846e+07,2.008861e+07,NaN,9573.632756,398.899256,5.046059e+09,5.065150e+09,5.062263e+09,2.008846e+07,NaN,28.979228,1057.058844,90.028904,1.590313,5.582895,2.008861e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5901.022129,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,2.859526e+11,0.031896,7.587457e+03,7.559295e+03,NaN,9315.073232,3663.463023,2.931521e+09,2.930776e+09,2.906374e+09,7.587081e+03,NaN,75.606458,29.650916,1033.615068,40.163847,6.284463,7.558797e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3035.861005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,1.960112e+14,1.000000,2.007113e+07,2.008010e+07,NaN,-8000.000000,0.000000,1.168381e+06,1.159725e+06,8.363090e+05,2.007113e+07,NaN,0.000000,1024.000000,0.000000,0.000000,0.000000,2.008010e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N



--- Unique Value Counts per Column ---


DESYNPUF_ID                       37780
CLM_ID                            66705
SEGMENT                               2
CLM_FROM_DT                        1120
CLM_THRU_DT                        1096
PRVDR_NUM                          2675
CLM_PMT_AMT                          92
NCH_PRMRY_PYR_CLM_PD_AMT             72
AT_PHYSN_NPI                      16670
OP_PHYSN_NPI                      12076
OT_PHYSN_NPI                       4386
CLM_ADMSN_DT                       1120
ADMTNG_ICD9_DGNS_CD                2285
CLM_PASS_THRU_PER_DIEM_AMT           15
NCH_BENE_IP_DDCTBL_AMT                3
NCH_BENE_PTA_COINSRNC_LBLTY_AM       36
NCH_BENE_BLOOD_DDCTBL_LBLTY_AM       18
CLM_UTLZTN_DAY_CNT                   99
NCH_BENE_DSCHRG_DT                 1096
CLM_DRG_CD                          739
ICD9_DGNS_CD_1                     2740
ICD9_DGNS_CD_2                     2961
ICD9_DGNS_CD_3                     2909
ICD9_DGNS_CD_4                     2907
ICD9_DGNS_CD_5                     2915


Part 2: Data Preprocessing and Exploration

2.1 Data Cleaning

In [10]:
# Create a copy of the original dataframe to preserve it during cleaning
df_cleaned = df.copy()

# 1. Drop Empty Columns
# Find all columns that start with 'HCPCS_CD_'
hcpcs_cols_to_drop = [col for col in df_cleaned.columns if col.startswith('HCPCS_CD_')]
df_cleaned.drop(columns=hcpcs_cols_to_drop, inplace=True)
print(f"Dropped {len(hcpcs_cols_to_drop)} empty HCPCS columns.")

# 2. Handle Missing Values
# Drop rows with missing essential identifiers/dates
essential_cols = ['CLM_ID', 'DESYNPUF_ID', 'CLM_FROM_DT', 'CLM_THRU_DT', 'PRVDR_NUM', 'CLM_UTLZTN_DAY_CNT']
df_cleaned.dropna(subset=essential_cols, inplace=True)
print(f"Shape after dropping rows with missing essential info: {df_cleaned.shape}")

# Fill missing optional NPIs with a placeholder string 'missing'
npi_cols = ['AT_PHYSN_NPI', 'OP_PHYSN_NPI', 'OT_PHYSN_NPI']
df_cleaned[npi_cols] = df_cleaned[npi_cols].fillna('missing')
print("Filled missing NPIs with the 'missing' placeholder.")

# 3. Correct Data Types
# Convert date columns from float/int to datetime objects
date_cols = ['CLM_FROM_DT', 'CLM_THRU_DT', 'CLM_ADMSN_DT', 'NCH_BENE_DSCHRG_DT']
for col in date_cols:
    # The conversion handles both float and int types by first casting to int, then to string.
    df_cleaned[col] = pd.to_datetime(df_cleaned[col].astype(int).astype(str), format='%Y%m%d')

# Convert NPI columns to string type, handling the mixed types (float and our 'missing' string)
for col in npi_cols:
    df_cleaned[col] = df_cleaned[col].apply(lambda x: str(int(x)) if x != 'missing' else x)
print("Corrected data types for date and NPI columns.")

# 4. Address Invalid Values
# Correct negative payment amounts by taking the absolute value
df_cleaned['CLM_PMT_AMT'] = df_cleaned['CLM_PMT_AMT'].abs()
print(f"Corrected negative values. Minimum CLM_PMT_AMT is now: {df_cleaned['CLM_PMT_AMT'].min()}")

# 5. Reset Index
# This is crucial for ensuring a clean, contiguous index for mapping later on.
df_cleaned.reset_index(drop=True, inplace=True)
print("DataFrame index has been reset.")

# --- Final Verification ---
print("\n--- Cleaned DataFrame Info ---")
df_cleaned.info()

print("\n\n--- First 5 Rows of Cleaned Data ---")
display(df_cleaned.head())

Dropped 45 empty HCPCS columns.
Shape after dropping rows with missing essential info: (66705, 36)
Filled missing NPIs with the 'missing' placeholder.
Corrected data types for date and NPI columns.
Corrected negative values. Minimum CLM_PMT_AMT is now: 0.0
DataFrame index has been reset.

--- Cleaned DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66705 entries, 0 to 66704
Data columns (total 36 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   DESYNPUF_ID                     66705 non-null  object        
 1   CLM_ID                          66705 non-null  int64         
 2   SEGMENT                         66705 non-null  int64         
 3   CLM_FROM_DT                     66705 non-null  datetime64[ns]
 4   CLM_THRU_DT                     66705 non-null  datetime64[ns]
 5   PRVDR_NUM                       66705 non-null  object        
 6   CLM_PMT_AMT          

,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,OT_PHYSN_NPI,CLM_ADMSN_DT,ADMTNG_ICD9_DGNS_CD,CLM_PASS_THRU_PER_DIEM_AMT,NCH_BENE_IP_DDCTBL_AMT,NCH_BENE_PTA_COINSRNC_LBLTY_AM,NCH_BENE_BLOOD_DDCTBL_LBLTY_AM,CLM_UTLZTN_DAY_CNT,NCH_BENE_DSCHRG_DT,CLM_DRG_CD,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_DGNS_CD_3,ICD9_DGNS_CD_4,ICD9_DGNS_CD_5,ICD9_DGNS_CD_6,ICD9_DGNS_CD_7,ICD9_DGNS_CD_8,ICD9_DGNS_CD_9,ICD9_DGNS_CD_10,ICD9_PRCDR_CD_1,ICD9_PRCDR_CD_2,ICD9_PRCDR_CD_3,ICD9_PRCDR_CD_4,ICD9_PRCDR_CD_5,ICD9_PRCDR_CD_6
0,00013D2EFD8E45D1,196661176988405,1,2010-03-12,2010-03-13,2600GD,4000.0,0.0,3139083564,missing,missing,2010-03-12,4580,0.0,1100.0,0.0,0.0,1.0,2010-03-13,217,7802,78820,V4501,4280,2720,4019,V4502,73300,E9330,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00016F745862898F,196201177000368,1,2009-04-12,2009-04-18,3900MB,26000.0,0.0,6476809087,missing,missing,2009-04-12,7866,0.0,1068.0,0.0,0.0,6.0,2009-04-18,201,1970,4019,5853,7843,2768,71590,2724,19889,5849,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00016F745862898F,196661177015632,1,2009-08-31,2009-09-02,3900HM,5000.0,0.0,611998537,611998537,missing,2009-08-31,6186,0.0,1068.0,0.0,0.0,2.0,2009-09-02,750,6186,2948,56400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7092.0,6186,V5866,NaN,NaN,NaN
3,00016F745862898F,196091176981058,1,2009-09-17,2009-09-20,3913XU,5000.0,0.0,4971602784,missing,1119000316,2009-09-17,29590,0.0,1068.0,0.0,0.0,3.0,2009-09-20,883,29623,30390,71690,34590,V1581,32723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00016F745862898F,196261176983265,1,2010-06-26,2010-07-01,3900MB,16000.0,0.0,6408400473,1960859579,missing,2010-06-26,5849,0.0,1100.0,0.0,0.0,5.0,2010-07-01,983,3569,4019,3542,V8801,78820,2639,7840,7856,4271,NaN,NaN,E8889,NaN,NaN,NaN,NaN


2.2 Exploratory Data Analysis (EDA)

In [11]:
# --- 1. Distribution of Key Numerical Features ---
print("--- Analyzing Numerical Feature Distributions ---")

# Plotting with Plotly for interactive and clean visuals
fig_pmt = go.Figure(data=[go.Histogram(x=df_cleaned['NCH_PRMRY_PYR_CLM_PD_AMT'], nbinsx=50)])
fig_pmt.update_layout(
    title_text='Distribution of Primary Payer Paid Amount (NCH_PRMRY_PYR_CLM_PD_AMT)',
    xaxis_title_text='Paid Amount (in $)',
    yaxis_title_text='Frequency',
    bargap=0.1
)
fig_pmt.show()

fig_days = go.Figure(data=[go.Histogram(x=df_cleaned['CLM_UTLZTN_DAY_CNT'], nbinsx=50)])
fig_days.update_layout(
    title_text='Distribution of Claim Utilization Day Count (CLM_UTLZTN_DAY_CNT)',
    xaxis_title_text='Number of Days (Length of Stay)',
    yaxis_title_text='Frequency'
)
# The distribution has a long tail, so we can zoom in on the lower range where most data lies.
fig_days.update_xaxes(range=[0, 40])
fig_days.show()


# --- 2. Frequency of Key Categorical Features ---
print("\n--- Analyzing Categorical Feature Frequencies ---")

def plot_top_n(df, column, n=20, title=""):
    """Helper function to create an interactive bar plot of the top N most frequent items in a column."""
    # Calculate top N most frequent items
    top_n = df[column].value_counts().nlargest(n)
    
    fig = go.Figure(data=[go.Bar(x=top_n.index.astype(str), y=top_n.values, text=top_n.values, textposition='auto')])
    fig.update_layout(
        title_text=title,
        xaxis_title_text=column,
        yaxis_title_text='Frequency',
        xaxis={'categoryorder':'total descending'} # Order bars by frequency
    )
    fig.show()

# Plot top providers
plot_top_n(df_cleaned, 'PRVDR_NUM', n=20, title='Top 20 Most Frequent Providers (PRVDR_NUM)')

# Plot top attending physicians (excluding our 'missing' placeholder)
df_att_physician = df_cleaned[df_cleaned['AT_PHYSN_NPI'] != 'missing']
plot_top_n(df_att_physician, 'AT_PHYSN_NPI', n=20, title='Top 20 Most Frequent Attending Physicians (AT_PHYSN_NPI)')

# Plot top admitting diagnoses
plot_top_n(df_cleaned, 'ADMTNG_ICD9_DGNS_CD', n=20, title='Top 20 Most Frequent Admitting Diagnoses')

--- Analyzing Numerical Feature Distributions ---



--- Analyzing Categorical Feature Frequencies ---
